In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from sklearn.datasets import load_breast_cancer

In [2]:
data = load_breast_cancer()

In [3]:
x= data['data']

In [4]:
y= data['target']

In [5]:
x.shape


(569, 30)

In [6]:
x_train, x_test, y_train, y_test= train_test_split(x, y, random_state= 42, train_size= 0.8)

In [7]:
scaler= StandardScaler()
x_train= scaler.fit_transform(x_train)
x_test= scaler.transform(x_test)

In [8]:
# converting numpy array into torch tensors
x_train_tensor= torch.from_numpy(x_train)
x_test_tensor= torch.from_numpy(x_test)
y_train_tensor= torch.from_numpy(y_train)
y_test_tensor= torch.from_numpy(y_test)

In [9]:
x_train_tensor.shape

torch.Size([455, 30])

In [10]:
x_train_tensor

tensor([[-1.4408, -0.4353, -1.3621,  ...,  0.9320,  2.0972,  1.8865],
        [ 1.9741,  1.7330,  2.0917,  ...,  2.6989,  1.8912,  2.4978],
        [-1.4000, -1.2496, -1.3452,  ..., -0.9702,  0.5976,  0.0579],
        ...,
        [ 0.0488, -0.5550, -0.0651,  ..., -1.2390, -0.7086, -1.2715],
        [-0.0390,  0.1021, -0.0314,  ...,  1.0500,  0.4343,  1.2134],
        [-0.5486,  0.3133, -0.6035,  ..., -0.6110, -0.3345, -0.8463]],
       dtype=torch.float64)

### Defining the model

In [11]:
epochs= 25
lr= 0.1

In [21]:
class MySimpleNN():
    def __init__ (self, x):
        self.weights= torch.rand(size=( x.shape[1], 1), dtype= torch.float64, requires_grad= True)
        self.bias= torch.zeros(1, dtype= torch.float64, requires_grad= True)

    def forward(self, x):
        z= torch.matmul(x, self.weights)+ self.bias
        y_pred= torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
        eps = 1e-7
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        return (-y*torch.log(y_pred) - (1-y)*torch.log(1-y_pred)).mean()

In [24]:
model= MySimpleNN(x_train_tensor)

In [25]:
# create training loop

for i in range (epochs):
    
    y_pred= model.forward(x_train_tensor)

    #loss
    loss= model.loss_function(y_pred, y_train_tensor)


    #back prop
    loss.backward()

    with torch.no_grad():
        model.weights-= lr* model.weights.grad
        model.bias-= lr* model.bias.grad

    print(f"Epoch: {i+1}; Loss: {loss.item():.2f}")

    model.weights.grad.zero_()
    model.bias.grad.zero_()




Epoch: 1; Loss: 3.43
Epoch: 2; Loss: 3.26
Epoch: 3; Loss: 3.10
Epoch: 4; Loss: 2.93
Epoch: 5; Loss: 2.75
Epoch: 6; Loss: 2.58
Epoch: 7; Loss: 2.40
Epoch: 8; Loss: 2.23
Epoch: 9; Loss: 2.06
Epoch: 10; Loss: 1.89
Epoch: 11; Loss: 1.73
Epoch: 12; Loss: 1.58
Epoch: 13; Loss: 1.44
Epoch: 14; Loss: 1.31
Epoch: 15; Loss: 1.20
Epoch: 16; Loss: 1.11
Epoch: 17; Loss: 1.03
Epoch: 18; Loss: 0.96
Epoch: 19; Loss: 0.91
Epoch: 20; Loss: 0.87
Epoch: 21; Loss: 0.84
Epoch: 22; Loss: 0.82
Epoch: 23; Loss: 0.81
Epoch: 24; Loss: 0.79
Epoch: 25; Loss: 0.78


In [28]:
with torch.no_grad():
    y_pred= model.forward(x_test_tensor)

In [34]:
y_pred= torch.where(
    y_pred >= 0.5,
    1.0,
    0.0
)

In [35]:
y_pred[: 5]

tensor([[0.],
        [0.],
        [0.],
        [1.],
        [1.]])

In [41]:
y_pred= y_pred.to(dtype= torch.float64)

In [45]:
accuracy= torch.where(
    y_pred== y_test_tensor,
    1,
    0
).to(dtype= torch.float64).mean()

In [46]:
accuracy

tensor(0.5302, dtype=torch.float64)